# Walmart Sales Forecasting — XGBoost

**მოდელი:** XGBoost Regressor
**კატეგორია:** Tree-based (Gradient Boosting)
**MLflow ექსპერიმენტი:** `XGBoost_Training`
**Runs:**
1. `XGBoost_EDA` — exploratory data analysis
2. `XGBoost_Cleaning` — preprocessing pipeline validation
3. `XGBoost_Feature_Selection` — feature set comparison
4. `XGBoost_CrossValidation` — TimeSeriesSplit CV
5. `XGBoost_GridSearch` — 10-iteration grid search
6. `XGBoost_Final` — საუკეთესო Pipeline (raw → prediction)

**მეტრიკა:** WMAE (Weighted Mean Absolute Error) — competition-ის ოფიციალური. სადღესასწაულო კვირებს ×5 წონა.

## 1. Setup

In [ ]:
# ინსტალაცია
!pip install xgboost mlflow dagshub --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature

print(f"XGBoost version: {xgb.__version__}")
print(f"MLflow version: {mlflow.__version__}")

In [ ]:
# Drive mount
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
print(f"Data directory: {DATA_DIR}")

In [ ]:
# DagsHub + MLflow setup
import dagshub

DAGSHUB_USERNAME = "zberi23"
DAGSHUB_REPO = "walmart-forecasting"

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)

EXPERIMENT_NAME = "XGBoost_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 2. Raw Data Loading

Pipeline იმუშავებს raw test set-ზე. ამიტომ სამივე original ფაილს ვტვირთავთ, არა preprocessed CSV-ს.

In [ ]:
# Raw მონაცემები — Pipeline-ის მოთხოვნის დასაცავად
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
stores_raw = pd.read_csv(f'{DATA_DIR}/stores.csv')
features_raw = pd.read_csv(f'{DATA_DIR}/features.csv.zip')

for df in [train_raw, test_raw, features_raw]:
    df['Date'] = pd.to_datetime(df['Date'])

print(f"Train raw: {train_raw.shape}")
print(f"Test raw:  {test_raw.shape}")
print(f"Stores:    {stores_raw.shape}")
print(f"Features:  {features_raw.shape}")

train_raw.head(3)

## 3. Exploratory Data Analysis

ვაანალიზებთ სამ ფაილს ცალ-ცალკე და ერთად, სანამ preprocessing pipeline-ს ავაშენებთ. მიზანია გავიგოთ განაწილებები, missing values-ის სტრუქტურა, სეზონურობა და holiday ეფექტი — რაც პირდაპირ განსაზღვრავს ქვემოთ არსებულ feature engineering-ის არჩევანს.

In [ ]:
eda = train_raw.merge(stores_raw, on='Store', how='left') \
                .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')

print("Merged EDA frame:", eda.shape)
eda.describe(include='all').T.head(20)

### 3.1 Missing values

In [ ]:
missing = eda.isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

plt.figure(figsize=(10, 5))
plt.barh(missing.index[::-1], missing.values[::-1])
plt.xlabel('% missing')
plt.title('Missing values by column')
plt.tight_layout()
plt.show()

print(missing)

MarkDown1–5 არის ყველაზე missing-heavy სვეტები (ეს პროდაქცია დაიწყო მხოლოდ 2011 ნოემბრიდან), CPI/Unemployment-ს რამდენიმე ხარვეზი აქვს store-level დროის ღერძზე.

### 3.2 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(eda['Weekly_Sales'], bins=100)
axes[0].set_title('Weekly_Sales distribution')
axes[0].set_xlabel('Weekly_Sales')

axes[1].hist(np.log1p(eda['Weekly_Sales'].clip(lower=0)), bins=100)
axes[1].set_title('log1p(Weekly_Sales) distribution')
axes[1].set_xlabel('log1p(Weekly_Sales)')
plt.tight_layout()
plt.show()

print(eda['Weekly_Sales'].describe())
print(f"Negative Weekly_Sales rows: {(eda['Weekly_Sales'] < 0).sum()}")

### 3.3 Seasonality and holiday effect

In [ ]:
weekly_avg = eda.groupby('Date')['Weekly_Sales'].mean()

plt.figure(figsize=(14, 5))
plt.plot(weekly_avg.index, weekly_avg.values)
holiday_dates = eda.loc[eda['IsHoliday'], 'Date'].unique()
for d in holiday_dates:
    plt.axvline(pd.Timestamp(d), color='red', alpha=0.15)
plt.title('Average Weekly_Sales over time (red = holiday week)')
plt.xlabel('Date')
plt.ylabel('Avg Weekly_Sales')
plt.tight_layout()
plt.show()

print(eda.groupby('IsHoliday')['Weekly_Sales'].agg(['mean', 'median', 'count']))

### 3.4 Store type / size / department effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

eda.groupby('Type')['Weekly_Sales'].mean().plot(kind='bar', ax=axes[0], title='Avg sales by store Type')
axes[1].scatter(eda['Size'], eda['Weekly_Sales'], s=2, alpha=0.15)
axes[1].set_title('Size vs Weekly_Sales')
axes[1].set_xlabel('Size')

top_depts = eda.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False).head(15)
top_depts.plot(kind='bar', ax=axes[2], title='Top 15 Dept by avg sales')

plt.tight_layout()
plt.show()

### 3.5 Correlation between numeric covariates

In [ ]:
num_cols = ['Weekly_Sales', 'Size', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
            'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
corr = eda[num_cols].corr()

plt.figure(figsize=(9, 7))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im)
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

corr['Weekly_Sales'].sort_values(ascending=False)

**დასკვნები EDA-დან:**
- Weekly_Sales ძლიერად skewed არის, Store/Dept/Size ცვლადები ყველაზე მეტ ვარიაციას ხსნიან.
- Holiday კვირები საშუალოდ უფრო მაღალი გაყიდვებით გამოირჩევა — ეს ამართლებს competition-ის ×5 წონას.
- ეკონომიკური ცვლადები (CPI, Fuel_Price, Unemployment, Temperature) სუსტად კორელირებენ სამიზნესთან პირდაპირ, მაგრამ store/dept-ის კონტექსტში მაინც სასარგებლო სიგნალს იძლევიან.
- Dept დონეზე დიდი განსხვავებაა საშუალო გაყიდვებში — ეს განაპირობებს Dept-დონის სტატისტიკური ფიჩერების დამატებას ქვემოთ.

## 4. Custom Preprocessor

sklearn-compatible Transformer, რომელიც Pipeline-ის შიგნით raw test set-ს (`Store, Dept, Date, IsHoliday`) გარდაქმნის მოდელისთვის მზა ფიჩერების მატრიცად. ყველა ახალი ცვლადი აგებულია მხოლოდ `Date`-სა და store/feature-level სტატიკურ ინფორმაციაზე დაყრდნობით, ისე რომ `transform` არასდროს არ საჭიროებს სამიზნე სვეტს (`Weekly_Sales`).

In [ ]:
class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    """
    Custom Pipeline transformer, რომელიც raw Walmart CSV-ებს გარდაქმნის
    მოდელისთვის მზა ფიჩერების მატრიცად.
    """

    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df.copy()
        self.features_df = features_df.copy()

        if not pd.api.types.is_datetime64_any_dtype(self.features_df['Date']):
            self.features_df['Date'] = pd.to_datetime(self.features_df['Date'])

        # Dept-level popularity, from static frequency in the store list only
        # (used purely as a stable identifier, not derived from Weekly_Sales)
        self._dept_rank_map = None

    def fit(self, X, y=None):
        dept_counts = X['Dept'].value_counts()
        self._dept_rank_map = {d: r for r, d in enumerate(dept_counts.index)}

        if y is None and 'Weekly_Sales' in X.columns:
            y = X['Weekly_Sales'].values

        hist = X[['Store', 'Dept', 'Date']].copy()
        if not pd.api.types.is_datetime64_any_dtype(hist['Date']):
            hist['Date'] = pd.to_datetime(hist['Date'])
        hist['Weekly_Sales'] = np.asarray(y, dtype=float)
        hist = hist.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        grp = hist.groupby(['Store', 'Dept'])['Weekly_Sales']
        hist['lag_1'] = grp.shift(1)
        hist['lag_52'] = grp.shift(52)
        hist['roll_4_mean'] = grp.transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
        hist['roll_12_mean'] = grp.transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())

        global_mean = hist['Weekly_Sales'].mean()
        self._global_fallback = {
            'lag_1': global_mean,
            'lag_52': global_mean,
            'roll_4_mean': global_mean,
            'roll_12_mean': global_mean,
        }
        self._history_features = hist[['Store', 'Dept', 'Date', 'lag_1', 'lag_52',
                                        'roll_4_mean', 'roll_12_mean']].sort_values('Date').reset_index(drop=True)
        return self

    def transform(self, X):
        df = X.copy()

        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        # ---- Merge ----
        df = df.merge(self.features_df, on=['Store', 'Date', 'IsHoliday'], how='left')
        df = df.merge(self.stores_df, on='Store', how='left')

        # ---- Missing values ----
        for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
            df[col] = df[col].fillna(0)

        for col in ['CPI', 'Unemployment', 'Temperature', 'Fuel_Price']:
            df[col] = df.groupby('Store')[col].transform(
                lambda x: x.fillna(method='ffill').fillna(method='bfill')
            )

        # ---- Calendar features ----
        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['Day_of_Year'] = df['Date'].dt.dayofyear
        df['Quarter'] = df['Date'].dt.quarter
        df['WeekOfMonth'] = ((df['Date'].dt.day - 1) // 7 + 1).astype(int)
        df['Is_MonthStart'] = df['Date'].dt.is_month_start.astype(int)
        df['Is_MonthEnd'] = df['Date'].dt.is_month_end.astype(int)

        # ---- Holiday-specific flags ----
        super_bowl = pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'])
        labor_day = pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'])
        thanksgiving = pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'])
        christmas = pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])
        all_holidays = super_bowl.append(labor_day).append(thanksgiving).append(christmas)

        df['Is_SuperBowl'] = df['Date'].isin(super_bowl).astype(int)
        df['Is_LaborDay'] = df['Date'].isin(labor_day).astype(int)
        df['Is_Thanksgiving'] = df['Date'].isin(thanksgiving).astype(int)
        df['Is_Christmas'] = df['Date'].isin(christmas).astype(int)

        # Days to the nearest known holiday, computed purely from the calendar
        holiday_sorted = pd.Series(sorted(all_holidays))
        def days_to_nearest(d):
            diffs = (holiday_sorted - d).dt.days
            return diffs.abs().min()
        df['Days_To_Holiday'] = df['Date'].apply(days_to_nearest)

        # ---- Cyclical encoding ----
        df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
        df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
        df['Week_Sin'] = np.sin(2 * np.pi * df['Week'] / 52)
        df['Week_Cos'] = np.cos(2 * np.pi * df['Week'] / 52)

        # ---- Store / markdown derived features ----
        markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
        df['MarkDown_Total'] = df[markdown_cols].sum(axis=1)
        df['MarkDown_Active_Count'] = (df[markdown_cols] > 0).sum(axis=1)
        df['Size_Per_Dept_Rank'] = df['Dept'].map(self._dept_rank_map).fillna(-1)

        size_bins = [0, 50000, 100000, 150000, 200000, np.inf]
        df['Size_Bucket'] = pd.cut(df['Size'], bins=size_bins, labels=False)

        # ---- Categorical encoding ----
        type_mapping = {'A': 0, 'B': 1, 'C': 2}
        df['Type_Encoded'] = df['Type'].map(type_mapping)
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        lag_cols = ['lag_1', 'lag_52', 'roll_4_mean', 'roll_12_mean']
        left = df[['Store', 'Dept', 'Date']].sort_values('Date')
        merged_lags = pd.merge_asof(
            left,
            self._history_features,
            on='Date',
            by=['Store', 'Dept'],
            direction='backward'
        ).set_index(left.index).sort_index()
        for col in lag_cols:
            df[col] = merged_lags[col].fillna(self._global_fallback[col])

        df = df.drop(columns=['Date', 'Type'])
        return df


# ტესტი: Preprocessor მუშაობს raw მონაცემებზე?
preprocessor = WalmartPreprocessor(stores_raw, features_raw)
X_test_transformed = preprocessor.fit(train_raw).transform(test_raw)
print(f"Test raw shape: {test_raw.shape}")
print(f"Test transformed: {X_test_transformed.shape}")
print(f"Columns: {list(X_test_transformed.columns)}")

## 5. WMAE Metric + მონაცემების მომზადება

In [ ]:
def wmae(y_true, y_pred, weights):
    """
    Weighted Mean Absolute Error — competition metric.
    სადღესასწაულო კვირებს წონა 5, სხვას 1.
    """
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def get_weights(is_holiday):
    """Holiday კვირას 5, სხვა 1."""
    return np.where(is_holiday == 1, 5, 1)


# Train/validation split — TIME-BASED (არა random!)
train_sorted = train_raw.sort_values('Date').reset_index(drop=True)
split_date = train_sorted['Date'].quantile(0.9)

X_train_raw = train_sorted[train_sorted['Date'] < split_date].drop(columns=['Weekly_Sales'])
y_train = train_sorted[train_sorted['Date'] < split_date]['Weekly_Sales'].values

X_val_raw = train_sorted[train_sorted['Date'] >= split_date].drop(columns=['Weekly_Sales'])
y_val = train_sorted[train_sorted['Date'] >= split_date]['Weekly_Sales'].values

print(f"Train: {X_train_raw.shape}, val: {X_val_raw.shape}")
print(f"Train date range: {X_train_raw['Date'].min()} → {X_train_raw['Date'].max()}")
print(f"Val date range:   {X_val_raw['Date'].min()} → {X_val_raw['Date'].max()}")

## 6. Run 1 — `XGBoost_Cleaning`

Preprocessing pipeline validation. `WalmartPreprocessor` fit-ავს Dept-frequency მეპინგს მხოლოდ train ნაწილზე და ცალ-ცალკე გარდაქმნის train/val/test-ს, რომ raw test-ზეც იმუშაოს უცვლელად.

In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    preprocessor = WalmartPreprocessor(stores_raw, features_raw)
    preprocessor.fit(X_train_raw, y_train)

    X_train_processed = preprocessor.transform(X_train_raw)
    X_val_processed = preprocessor.transform(X_val_raw)
    X_test_processed = preprocessor.transform(test_raw)

    train_nans = X_train_processed.isnull().sum().sum()
    val_nans = X_val_processed.isnull().sum().sum()
    test_nans = X_test_processed.isnull().sum().sum()

    mlflow.log_param("train_rows_raw", X_train_raw.shape[0])
    mlflow.log_param("train_rows_processed", X_train_processed.shape[0])
    mlflow.log_param("test_rows_raw", test_raw.shape[0])
    mlflow.log_param("test_rows_processed", X_test_processed.shape[0])
    mlflow.log_param("n_features_after_preprocessing", X_train_processed.shape[1])
    mlflow.log_param("markdown_fill_value", 0)
    mlflow.log_param("cpi_unemployment_temp_fuel_fill_method", "groupby_store_ffill_bfill")
    mlflow.log_param("features", list(X_train_processed.columns))

    mlflow.log_metric("train_missing_after_clean", train_nans)
    mlflow.log_metric("val_missing_after_clean", val_nans)
    mlflow.log_metric("test_missing_after_clean", test_nans)

    mlflow.set_tag("stage", "cleaning")

    print(f"Train NaN after clean: {train_nans}")
    print(f"Val NaN after clean:   {val_nans}")
    print(f"Test NaN after clean:  {test_nans}")
    print(f"Features: {X_train_processed.shape[1]}")

## 7. Run 2 — `XGBoost_Feature_Selection`

სხვადასხვა feature set-ის შედარება baseline XGBoost-ით:

- **all** — ყველა ფიჩერი
- **no_markdown** — MarkDown-ების გარეშე
- **core** — მხოლოდ ძირითადი date + store ფიჩერები
- **extended** — core + ახალი calendar/markdown/size ფიჩერები (Days_To_Holiday, WeekOfMonth, MarkDown_Total, Size_Bucket, Size_Per_Dept_Rank და სხვ.)

In [ ]:
FEATURE_SETS = {
    "all": list(X_train_processed.columns),

    "no_markdown": [c for c in X_train_processed.columns
                    if c not in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']],

    "core": ['Store', 'Dept', 'IsHoliday', 'Size', 'Type_Encoded',
             'Year', 'Month', 'Week', 'Day_of_Year',
             'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas',
             'Month_Sin', 'Month_Cos', 'Week_Sin', 'Week_Cos'],

    "extended": ['Store', 'Dept', 'IsHoliday', 'Size', 'Type_Encoded',
                 'Year', 'Month', 'Week', 'Day_of_Year', 'Quarter',
                 'WeekOfMonth', 'Is_MonthStart', 'Is_MonthEnd',
                 'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas',
                 'Days_To_Holiday', 'Month_Sin', 'Month_Cos', 'Week_Sin', 'Week_Cos',
                 'MarkDown_Total', 'MarkDown_Active_Count',
                 'Size_Bucket', 'Size_Per_Dept_Rank',
                 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
                 'lag_1', 'lag_52', 'roll_4_mean', 'roll_12_mean'],
}

BASELINE_PARAMS = {
    'n_estimators': 200,
    'max_depth': 8,
    'learning_rate': 0.1,
    'random_state': 42,
    'tree_method': 'hist',
    'n_jobs': -1,
}

feature_selection_results = {}

with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_param("feature_sets_tested", list(FEATURE_SETS.keys()))

    for fs_name, feats in FEATURE_SETS.items():
        with mlflow.start_run(run_name=f"FS_{fs_name}", nested=True):
            X_tr = X_train_processed[feats]
            X_va = X_val_processed[feats]

            model = xgb.XGBRegressor(**BASELINE_PARAMS)
            model.fit(X_tr, y_train, eval_set=[(X_va, y_val)], verbose=False)

            preds = model.predict(X_va)
            weights = get_weights(X_val_processed['IsHoliday'].values)
            score = wmae(y_val, preds, weights)

            feature_selection_results[fs_name] = score

            mlflow.log_param("feature_set", fs_name)
            mlflow.log_param("n_features", len(feats))
            mlflow.log_metric("val_wmae", score)

            print(f"[{fs_name:12}] n_features={len(feats):3d}, WMAE = {score:.2f}")

    best_fs = min(feature_selection_results, key=feature_selection_results.get)
    mlflow.log_metric("best_val_wmae", feature_selection_results[best_fs])
    mlflow.log_param("best_feature_set", best_fs)
    mlflow.set_tag("stage", "feature_selection")

    print(f"\nBest feature set: {best_fs} (WMAE={feature_selection_results[best_fs]:.2f})")

BEST_FEATURES = FEATURE_SETS[best_fs]

## 8. Run 3 — `XGBoost_CrossValidation`

**TimeSeriesSplit** — time-series-ისთვის სპეციფიკური CV. K-Fold-ს არ ვიყენებთ, რადგან შემთხვევითი shuffle დროის თანმიმდევრობას არღვევს.

TimeSeriesSplit ინახავს დროის თანმიმდევრობას: fold-1 = train[0:20%], val[20:40%]; fold-2 = train[0:40%], val[40:60%]; ა.შ.

თითოეულ fold-ში `WalmartPreprocessor` თავიდან fit-დება მხოლოდ იმ fold-ის training slice-ზე, რომ lag/rolling ფიჩერები არასდროს ხედავდნენ მომავალ fold-ებს.

In [ ]:
N_SPLITS = 5

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cv_scores = []

with mlflow.start_run(run_name="XGBoost_CrossValidation"):
    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_param("cv_method", "TimeSeriesSplit")
    mlflow.log_param("n_splits", N_SPLITS)
    mlflow.log_param("feature_set", best_fs)

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_raw)):
        X_tr_raw_fold = X_train_raw.iloc[tr_idx]
        y_tr_fold = y_train[tr_idx]
        X_va_raw_fold = X_train_raw.iloc[va_idx]
        y_va_fold = y_train[va_idx]

        fold_preprocessor = WalmartPreprocessor(stores_raw, features_raw)
        fold_preprocessor.fit(X_tr_raw_fold, y_tr_fold)

        X_tr_fold = fold_preprocessor.transform(X_tr_raw_fold)[BEST_FEATURES]
        X_va_fold = fold_preprocessor.transform(X_va_raw_fold)[BEST_FEATURES]

        model = xgb.XGBRegressor(**BASELINE_PARAMS)
        model.fit(X_tr_fold, y_tr_fold, verbose=False)

        preds = model.predict(X_va_fold)
        weights_fold = get_weights(X_va_raw_fold['IsHoliday'].values.astype(int))
        score = wmae(y_va_fold, preds, weights_fold)

        cv_scores.append(score)
        mlflow.log_metric("fold_wmae", score, step=fold)
        print(f"Fold {fold+1}: WMAE = {score:.2f}")

    mean_score = np.mean(cv_scores)
    std_score = np.std(cv_scores)

    mlflow.log_metric("cv_wmae_mean", mean_score)
    mlflow.log_metric("cv_wmae_std", std_score)
    mlflow.set_tag("stage", "cross_validation")

    print(f"\nCV WMAE: {mean_score:.2f} ± {std_score:.2f}")

## 9. Run 4 — `XGBoost_GridSearch`

10 კომბინაციის grid search (`max_depth` × `learning_rate`, 5×2), თითოეული ცალკე nested run-ად ილოგება. შეფასება ხდება დროით მოწესრიგებულ hold-out ვალიდაციაზე (`X_val_processed`), იმავე train-ზე გაწვრთნით რაზეც feature selection და CV ეყრდნობოდა.

In [ ]:
GRID = {
    'max_depth': [4, 6, 8, 10, 12],
    'learning_rate': [0.05, 0.1],
}
FIXED_PARAMS = {
    'n_estimators': 300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'tree_method': 'hist',
    'n_jobs': -1,
}

param_combinations = list(ParameterGrid(GRID))
print(f"Total grid search iterations: {len(param_combinations)}")

grid_results = []

with mlflow.start_run(run_name="XGBoost_GridSearch"):
    mlflow.log_param("grid", GRID)
    mlflow.log_param("fixed_params", FIXED_PARAMS)
    mlflow.log_param("n_iterations", len(param_combinations))
    mlflow.log_param("feature_set", best_fs)

    for i, combo in enumerate(param_combinations):
        params = {**FIXED_PARAMS, **combo}

        with mlflow.start_run(run_name=f"Grid_{i+1}", nested=True):
            model = xgb.XGBRegressor(**params)
            model.fit(X_train_processed[BEST_FEATURES], y_train, verbose=False)

            preds = model.predict(X_val_processed[BEST_FEATURES])
            weights = get_weights(X_val_processed['IsHoliday'].values)
            score = wmae(y_val, preds, weights)

            grid_results.append({**combo, 'val_wmae': score})

            mlflow.log_params(params)
            mlflow.log_metric("val_wmae", score)
            print(f"[{i+1:2d}/{len(param_combinations)}] {combo} -> WMAE = {score:.2f}")

    best_combo = min(grid_results, key=lambda r: r['val_wmae'])
    BEST_PARAMS = {**FIXED_PARAMS, **{k: v for k, v in best_combo.items() if k != 'val_wmae'}}

    mlflow.log_metric("best_val_wmae", best_combo['val_wmae'])
    mlflow.log_params(BEST_PARAMS)
    mlflow.set_tag("stage", "grid_search")

    print(f"\nBest params: {BEST_PARAMS}")
    print(f"Best WMAE:   {best_combo['val_wmae']:.2f}")

## 10. Run 5 — `XGBoost_Final` (Pipeline)

საბოლოო sklearn Pipeline:

```
raw test.csv (Store, Dept, Date, IsHoliday)
         ↓
WalmartPreprocessor  (merge + fillna + FE)
         ↓
FeatureSelector      (best feature subset)
         ↓
XGBoost              (best hyperparameters)
         ↓
predictions
```

პირველ რიგში ვზომავთ generalization-ს train-ზე გაწვრთნილი პაიპლაინით hold-out ვალიდაციაზე (`val_wmae`). ამის შემდეგ ცალკე ვაწვრთნით საბოლოო production pipeline-ს მთელ მონაცემზე (train+val) deployment-ისთვის — ამ ვერსიის შედეგი ცალკე მეტრიკადაა ჩაწერილი (`full_data_train_wmae`), რომ არ აგვერიოს hold-out შეფასებაში.

In [ ]:
class FeatureSelector(BaseEstimator, TransformerMixin):
    """Pipeline-compatible feature subset selector."""
    def __init__(self, features):
        self.features = features
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.features]


eval_pipeline = Pipeline([
    ('preprocess', WalmartPreprocessor(stores_raw, features_raw)),
    ('select', FeatureSelector(BEST_FEATURES)),
    ('model', xgb.XGBRegressor(**BEST_PARAMS)),
])

X_full_raw = train_sorted.drop(columns=['Weekly_Sales'])
y_full = train_sorted['Weekly_Sales'].values

print("Pipeline steps:")
for name, step in eval_pipeline.steps:
    print(f"  {name}: {type(step).__name__}")

In [ ]:
with mlflow.start_run(run_name="XGBoost_Final") as final_run:
    # Honest hold-out evaluation: fit only on the train slice, score on val
    eval_pipeline.fit(X_train_raw, y_train)
    val_preds = eval_pipeline.predict(X_val_raw)
    val_weights = get_weights(preprocessor.transform(X_val_raw)['IsHoliday'].values)
    val_wmae = wmae(y_val, val_preds, val_weights)

    # Production pipeline: refit on all available labelled data for deployment
    final_pipeline = Pipeline([
        ('preprocess', WalmartPreprocessor(stores_raw, features_raw)),
        ('select', FeatureSelector(BEST_FEATURES)),
        ('model', xgb.XGBRegressor(**BEST_PARAMS)),
    ])
    final_pipeline.fit(X_full_raw, y_full)

    full_fit_preds = final_pipeline.predict(X_full_raw)
    full_fit_weights = get_weights(final_pipeline.named_steps['preprocess'].transform(X_full_raw)['IsHoliday'].values)
    full_data_train_wmae = wmae(y_full, full_fit_preds, full_fit_weights)

    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("pipeline_steps", str([s[0] for s in final_pipeline.steps]))
    mlflow.log_param("feature_set", best_fs)
    mlflow.log_param("n_features_used", len(BEST_FEATURES))
    mlflow.log_param("training_rows", len(X_full_raw))

    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("full_data_train_wmae", full_data_train_wmae)

    signature = infer_signature(X_val_raw.head(100), val_preds[:100])

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="xgboost_pipeline",
        signature=signature,
        input_example=X_val_raw.head(5),
        registered_model_name="walmart_xgboost",
        serialization_format="cloudpickle"
    )

    mlflow.set_tag("stage", "final")
    mlflow.set_tag("model_type", "XGBoost")

    print(f"\nFinal Pipeline trained")
    print(f"Hold-out val WMAE:        {val_wmae:.2f}")
    print(f"Full-data train WMAE:     {full_data_train_wmae:.2f}")
    print(f"Registered as 'walmart_xgboost' in Model Registry")
    print(f"Run ID: {final_run.info.run_id}")

## 11. Pipeline-ის ვალიდაცია raw test-ზე

დავრწმუნდეთ, რომ Pipeline მუშაობს უცვლელ, preprocessed არა test set-ზე.

In [ ]:
test_predictions = final_pipeline.predict(test_raw)

print(f"Raw test shape: {test_raw.shape}")
print(f"Predictions shape: {test_predictions.shape}")
print(f"\nFirst 5 predictions:")
for i, p in enumerate(test_predictions[:5]):
    print(f"  Row {i}: Store={test_raw.iloc[i]['Store']}, "
          f"Dept={test_raw.iloc[i]['Dept']}, "
          f"Date={test_raw.iloc[i]['Date']}, "
          f"Predicted Weekly_Sales = {p:.2f}")

import joblib
import os
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
joblib.dump(final_pipeline, f'{PROJECT_DIR}/models/xgboost_pipeline.pkl')
print(f"\nPipeline შენახულია: {PROJECT_DIR}/models/xgboost_pipeline.pkl")

## 12. Feature Importance

In [ ]:
xgb_model = final_pipeline.named_steps['model']

importances = pd.DataFrame({
    'feature': BEST_FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(importances['feature'].head(20)[::-1], importances['importance'].head(20)[::-1])
plt.title('XGBoost Top 20 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(importances.head(10).to_string(index=False))

## 13. შეჯამება

XGBoost დასრულდა EDA-სა და 5 MLflow run-ით. Custom `WalmartPreprocessor` sklearn Transformer-ად აწყობილი Pipeline პირდაპირ raw CSV-ს იღებს და პროგნოზებს აბრუნებს.

**Pipeline-ის ეტაპები:**
- დეტალური EDA — target-ის განაწილება, missing values, სეზონურობა, Store Type/Size/Dept ეფექტები
- Feature Selection ოთხ ვარიანტს შორის (`all`, `no_markdown`, `core`, `extended`)
- 5-fold TimeSeriesSplit CV, სადაც თითოეული fold-ისთვის Preprocessor ცალკე fit-დება
- 10-იტერაციიანი Grid Search (`max_depth` × `learning_rate`)
- Hold-out ვალიდაცია + ცალკე production pipeline სრულ მონაცემზე

**შემდეგი:** N-BEATS, ARIMA/SARIMA, PatchTST notebook-ები. ბოლოს ყველა WMAE-ს ვადარებთ inference notebook-ში.